# Exercise 3: CrewAI Agent Crew as a NAT Function

A pre-built **CrewAI crew** with two specialized agents:
- **Analyst** - Examines input and extracts key insights
- **Writer** - Produces a structured report from the analysis

```
Input --> Analyst Agent --> Analysis --> Writer Agent --> Report
          (Bielik)                       (Bielik)
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Step 1: Examine the CrewAI Code

The implementation is in `workshop_agents/crewai_writing_crew.py`. Key differences from LangChain:
- CrewAI uses **role-based agents** (Analyst, Writer) instead of tool-based agents
- Agents work in a **sequential pipeline** (analyst first, then writer)
- CrewAI's `LLM` class connects to Bielik via vLLM with `openai/` prefix
- Same `@register_function` + `FunctionInfo.from_fn()` pattern

In [2]:
with open("../../workshop_agents/crewai_writing_crew.py") as f:
    print(f.read())

"""
CrewAI Writing Crew - registered as a NeMo Agent Toolkit Function.

This module creates a CrewAI crew with two agents:
- Analyst: Examines the input and extracts key insights
- Writer: Takes the analysis and produces a structured report

The crew is wrapped as a NAT Function for composition with other agents.
"""

from pydantic import Field

from nat.cli.register_workflow import register_function
from nat.builder.function_info import FunctionInfo
from nat.builder.builder import Builder
from nat.data_models.function import FunctionBaseConfig
from nat.data_models.component_ref import LLMRef


class CrewAIWritingCrewConfig(FunctionBaseConfig, name="crewai_writing_crew"):
    """Configuration for the CrewAI writing crew."""

    llm_ref: LLMRef = Field(description="Reference to the LLM to use")
    description: str = Field(default="Zespół CrewAI do analizy i pisania raportów")


@register_function(
    config_type=CrewAIWritingCrewConfig,
)
async def crewai_writing_crew(_config: CrewAI

## Step 2: Run the CrewAI Crew via NAT

In [3]:
!nat run --config_file config.yaml --input "Rodzina modeli językowych Bielik: polskie modele LLM zbudowane przez SpeakLeash, wykorzystujące destylację wiedzy i trenowane na wysokiej jakości polskich danych tekstowych."

2026-04-13 18:26:30 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 18:26:30 - ERROR    - nat.data_models.config:118 - Requested functions type `crewai_writing_crew` not found. Have you ensured the necessary package has been installed with `uv pip install`?
Available functions names:
 - nat.plugins.langchain/langgraph_wrapper
 - nat.plugins.langchain.tools/code_generation
 - nat.plugins.langchain.tools/tavily_internet_search
 - nat.plugins.langchain.tools/wiki_search
 - nat.plugins.langchain.agent.auto_memory_wrapper/auto_memory_agent
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_init
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_recombiner
 - nat.plugins.langchain.agent.react_agent/react_agent
 - nat.plugins.langchain.agent.react_agent/per_user_react_agent
 - nat.plugins.langchain.agent.reasoning_agent/reasoning_agent
 - nat.plugins.langchain.agent.responses_api_agent/responses_api_agent
 - nat.plugins.langchain.agent

## CrewAI vs LangChain

| Feature | LangChain ReAct | CrewAI Crew |
|---------|----------------|-------------|
| Pattern | Tool-calling loop | Role-based pipeline |
| Best for | Research, data gathering | Content creation, multi-step processing |
| Agents | Single agent, multiple tools | Multiple agents, specialized roles |
| Flow | Dynamic (agent decides) | Structured (sequential/hierarchical) |

**Key insight:** With NAT, you don't have to choose - you can compose both in a single workflow!

**Next:** Exercise 4 - combine the LangChain research agent and CrewAI writing crew.